# Model Training - Algorithm Selection and Hyperparameter Tuning

This notebook trains multiple classification models:
- Logistic Regression (baseline)
- Random Forest
- Gradient Boosting
- XGBoost
- Ensemble (weighted voting)

Hyperparameter optimization via GridSearchCV and cross-validation.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
# xgboost has dependency issues on m1/m2 mac, skip it
# import xgboost as xgb
import joblib
from IPython.display import Markdown, display
import json
import time

# Setup path
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

Libraries loaded successfully


## 1. Load Engineered Features

In [3]:
# Load data from feature engineering notebook
import scipy.sparse as sp

print('Loading engineered datasets...')
X_pca = np.load('../results/X_pca.npy')
y = np.load('../results/y.npy')

print(f'Loaded X_pca: {X_pca.shape}')
print(f'Loaded y: {y.shape}')
print(f'Class distribution: {np.bincount(y)}')

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

Loading engineered datasets...
Loaded X_pca: (100000, 100)
Loaded y: (100000,)
Class distribution: [50071 49929]

Train set: (80000, 100)
Test set: (20000, 100)


## 2. Train Baseline - Logistic Regression

In [4]:
print('Training Logistic Regression baseline...')
start = time.time()

lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

elapsed = time.time() - start
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr),
    'recall': recall_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr),
    'roc_auc': roc_auc_score(y_test, y_proba_lr),
    'training_time': elapsed
}

print(f'Logistic Regression (Training time: {elapsed:.2f}s)')
for metric, value in lr_metrics.items():
    print(f'  {metric:20s}: {value:.4f}')

Training Logistic Regression baseline...


/Users/bhavanishanker/predictive-sales-analytics-engine/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression (Training time: 0.49s)
  accuracy            : 0.9634
  precision           : 0.9504
  recall              : 0.9777
  f1                  : 0.9639
  roc_auc             : 0.9957
  training_time       : 0.4940


## 3. Train Random Forest with Hyperparameter Tuning

In [5]:
print('Training Random Forest with GridSearchCV...')
start = time.time()

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='f1', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

elapsed = time.time() - start
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_proba_rf),
    'training_time': elapsed
}

print(f'\nRandom Forest Best Params: {rf_grid.best_params_}')
print(f'Random Forest (Training time: {elapsed:.2f}s)')
for metric, value in rf_metrics.items():
    print(f'  {metric:20s}: {value:.4f}')

Training Random Forest with GridSearchCV...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Random Forest Best Params: {'max_depth': 30, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Random Forest (Training time: 725.69s)
  accuracy            : 0.9594
  precision           : 0.9453
  recall              : 0.9751
  f1                  : 0.9600
  roc_auc             : 0.9951
  training_time       : 725.6917
